# Principal component analysis (PCA)

Real data often has far more columns than it has independent things going on. Pixels, sensor readings and survey answers move together. PCA finds a small set of new axes that capture most of the variation, so you can plot, compress and denoise. It is the first thing to try when there are too many features, and it ties together the covariance matrices from probability and the SVD from linear algebra.

Order: intuition, maths, cost function, from-scratch version, library version, tuning, what breaks.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.model_selection import cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from common.datasets import load_digits_data, load_iris_flowers, load_wine_data, make_circles_data

SEED = 42
rng = np.random.default_rng(SEED)
np.set_printoptions(precision=3, suppress=True)
plt.rcParams["figure.figsize"] = (6, 3.5)

## 1. Intuition

Take a cloud of points that stretches along a diagonal, like height against weight. Shine a light from the side and look at the shadow on a line. Rotate the line and the shadow gets longer or shorter. The line along which the shadow is longest is the direction in which the data vary most. That is the **first principal component**. The best second direction is perpendicular to it, and so on.

Keep the first few and you have a lower-dimensional picture that loses as little spread as possible. The claim is that spread is information, and the directions with almost no spread are mostly noise or redundancy.

## 2. Maths and cost function

Centre the data so each column has mean zero, giving the matrix $X$ with $n$ rows and $d$ columns. Its covariance matrix is $S=\frac1{n-1}X^\top X$.

For a unit vector $\mathbf w$ the projections are $X\mathbf w$, and their variance is $\mathbf w^\top S\mathbf w$. PCA picks the $\mathbf w$ that maximises this with $\lVert\mathbf w\rVert=1$. Add a Lagrange multiplier and set the gradient to zero (calculus in `01_linear_algebra_calculus/03`):

$$S\mathbf w=\lambda\mathbf w$$

So the best direction is an **eigenvector** of the covariance matrix, and the variance it captures is the eigenvalue $\lambda$. The largest eigenvalue gives PC1, the next gives PC2, and the eigenvectors of a symmetric matrix are perpendicular, so the components are automatically uncorrelated.

**Cost function.** Two views give the same answer. One maximises the variance kept. The other minimises the **reconstruction error**, the squared distance between each point and its projection back into the original space. By Pythagoras, variance kept plus error is constant, so maximising one minimises the other. Keeping $k$ components leaves an average error equal to the sum of the discarded eigenvalues.

**Via the SVD** (see `01_linear_algebra_calculus/02_decompositions.ipynb`). Write $X=U\Sigma V^\top$. Then $X^\top X=V\Sigma^2V^\top$, so the columns of $V$ are the eigenvectors and the eigenvalues are $\lambda_i=\sigma_i^2/(n-1)$. The projected data (the **scores**) are $XV=U\Sigma$. The SVD never builds $X^\top X$, which is more accurate, so this is how libraries do it.

## 3. From scratch

First see the variance idea directly. Generate correlated 2D data, project onto a unit vector at each angle and measure the variance of the shadow.

In [ ]:
cov_true = np.array([[3.0, 2.2], [2.2, 2.0]])
Xt = rng.multivariate_normal([0, 0], cov_true, 400)
Xt -= Xt.mean(axis=0)
angles = np.linspace(0, np.pi, 181)
var_at = [np.var(Xt @ np.array([np.cos(a), np.sin(a)]), ddof=1) for a in angles]
best = angles[int(np.argmax(var_at))]
vals, vecs = np.linalg.eigh(np.cov(Xt.T))
print(
    "angle of max variance (deg):",
    round(np.degrees(best), 1),
    " eigenvector angle (deg):",
    round(np.degrees(np.arctan2(vecs[1, 1], vecs[0, 1])) % 180, 1),
)
print("max projected variance:", round(max(var_at), 3), " largest eigenvalue:", round(vals[-1], 3))
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].scatter(*Xt.T, s=10, alpha=0.5)
for j in (0, 1):
    axes[0].arrow(0, 0, *(2 * np.sqrt(vals[j]) * vecs[:, j]), width=0.04, color="r")
axes[0].set_aspect("equal")
axes[1].plot(np.degrees(angles), var_at)
axes[1].set(xlabel="projection angle (degrees)", ylabel="variance of the shadow")
plt.show()

The variance peaks at the angle of the top eigenvector and equals its eigenvalue. The red arrows are the two eigenvectors, which are perpendicular.

Now the full recipe as a function: centre, SVD, keep $k$ rows of $V^\top$.

In [ ]:
def pca_svd(X: np.ndarray, k: int):
    """Return the k components (rows), their variances, the mean and the scores."""
    mean = X.mean(axis=0)
    Xc = X - mean
    _, s, Vt = np.linalg.svd(Xc, full_matrices=False)
    variances = s**2 / (len(X) - 1)
    scores = Xc @ Vt[:k].T
    return Vt[:k], variances, mean, scores


Xi, yi = load_iris_flowers()
comps, variances, mean, scores = pca_svd(Xi.to_numpy(), 2)
ref = PCA(2).fit(Xi)
print("explained variance ratio, mine:   ", (variances / variances.sum())[:2])
print("explained variance ratio, sklearn:", ref.explained_variance_ratio_)
print("components equal up to sign:", np.allclose(np.abs(comps), np.abs(ref.components_)))
print("scores equal up to sign:", np.allclose(np.abs(scores), np.abs(ref.transform(Xi))))

The components match sklearn up to sign. The sign of an eigenvector is arbitrary (if $\mathbf w$ works then so does $-\mathbf w$), so never compare PCA outputs without allowing for it. Iris has four features and the first component alone keeps 92.5% of the variance.

In [ ]:
fig, ax = plt.subplots()
for c in (0, 1, 2):
    ax.scatter(*scores[yi == c].T, s=16, label=f"species {c}")
ax.set(xlabel="PC1", ylabel="PC2")
ax.legend()
plt.show()
print(pd.DataFrame(comps, columns=Xi.columns, index=["PC1", "PC2"]).round(2))

The rows of that table are the **loadings**: how much each original feature contributes to a component. PC1 leans on petal length mostly, which is the feature that separates the species. PCA did not use the labels, yet the species separate along PC1.

## 4. Library version and choosing how many components

The 8 by 8 digits have 64 pixel columns. How many components do we need? `explained_variance_ratio_` gives the share of variance each one carries. The cumulative sum is the share kept.

In [ ]:
Xd, yd = load_digits_data()
pca = PCA(random_state=SEED).fit(Xd)
cum = np.cumsum(pca.explained_variance_ratio_)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
axes[0].bar(range(1, 21), pca.explained_variance_ratio_[:20])
axes[0].set(xlabel="component", ylabel="explained variance ratio", title="scree plot (first 20)")
axes[1].plot(range(1, 65), cum)
axes[1].axhline(0.95, color="r", ls="--")
axes[1].set(xlabel="components kept", ylabel="cumulative variance", title="cumulative")
plt.show()
for target in (0.80, 0.90, 0.95, 0.99):
    print(
        f"{target:.0%} of the variance needs {int(np.argmax(cum >= target)) + 1} of 64 components"
    )

Common ways to choose:

- **A variance target** such as 90% or 95%. Simple and defensible. Here 95% needs 29 components, less than half of 64.
- **The elbow** in the scree plot, where the bars flatten. Subjective, but useful for a quick look.
- **Downstream performance.** If the goal is a classifier, pick the smallest number of components that keeps cross-validated accuracy (section 7, and notebook 06).

In scikit-learn, `PCA(n_components=0.95)` picks the count for you.

## 5. Reconstruction error and compression

Project to $k$ components, then map back with `inverse_transform`. What is lost is the reconstruction error. We can check the theory: the average squared error should equal the sum of the discarded eigenvalues.

In [ ]:
rows = []
for k in (2, 5, 10, 20, 40):
    p = PCA(k, random_state=SEED).fit(Xd)
    recon = p.inverse_transform(p.transform(Xd))
    mse_total = ((Xd.to_numpy() - recon) ** 2).sum(axis=1).mean()
    discarded = pca.explained_variance_[k:].sum() * (len(Xd) - 1) / len(Xd)
    rows.append(
        {
            "k": k,
            "mean squared error per image": mse_total,
            "sum of discarded eigenvalues": discarded,
            "stored numbers": f"{k * (len(Xd) + 64) / (len(Xd) * 64):.0%} of original",
        }
    )
print(pd.DataFrame(rows).round(2).to_string(index=False))
fig, axes = plt.subplots(6, 8, figsize=(10, 7.5))
sample = [0, 1, 2, 3, 4, 5, 6, 7]
for r, k in enumerate((None, 40, 20, 10, 5, 2)):
    if k is None:
        imgs = Xd.to_numpy()[sample]
    else:
        p = PCA(k, random_state=SEED).fit(Xd)
        imgs = p.inverse_transform(p.transform(Xd.iloc[sample]))
    for c in range(8):
        axes[r, c].imshow(imgs[c].reshape(8, 8), cmap="gray_r")
        axes[r, c].axis("off")
    axes[r, 0].set_title("original" if k is None else f"k={k}", loc="left", fontsize=9)
plt.show()

The two error columns match, as the theory said. By 20 components the digits are still recognisable, at 10 they are blurrier and at 2 only a coarse blob remains. At 20 components the stored numbers are about 32% of the original, so this is lossy compression that you tune with $k$.

## 6. Whitening

PCA scores are uncorrelated but have different variances: PC1's is large and PC20's is tiny. **Whitening** divides each score by its standard deviation so that every component has variance 1, and the covariance matrix becomes the identity. Some methods assume this (for example independent component analysis, and some distance-based methods on the scores).

In [ ]:
p = PCA(10, whiten=True, random_state=SEED).fit(Xd)
Z = p.transform(Xd)
print("score variances, plain PCA: ", PCA(10).fit_transform(Xd).var(axis=0, ddof=1).round(1))
print("score variances, whitened:  ", Z.var(axis=0, ddof=1).round(3))
print("off-diagonal covariance is ~0:", bool(np.abs(np.cov(Z.T) - np.eye(10)).max() < 1e-8))

Whitening has a cost. It scales the weak components up to the same size as the strong ones, and the weak ones are the ones that hold the most noise. Whiten only when you need to, and not past the point where the components stop carrying signal.

## 7. Choosing k by downstream accuracy

Does a classifier lose anything when it sees $k$ components instead of 64 pixels? Cross-validate kNN on the digits, with PCA inside the pipeline so each fold fits its own components.

In [ ]:
for k in (2, 5, 10, 20, 30, 64):
    model = make_pipeline(PCA(k, random_state=SEED), KNeighborsClassifier(5))
    print(f"{k:>2} components: CV accuracy {cross_val_score(model, Xd, yd, cv=5).mean():.3f}")

Accuracy reaches 95.8% at 20 components and 96.2% at 30, against 96.3% with all 64 pixels. The classifier is faster, because distances are cheaper and the noisy trailing directions are gone.

## 8. What breaks

**Scale.** PCA maximises variance, so a feature measured in large units wins by default. The wine data mix proline (hundreds to over a thousand) with columns near 1.

In [ ]:
Xw, yw = load_wine_data()
raw = PCA(2).fit(Xw)
scaled = PCA(2).fit(StandardScaler().fit_transform(Xw))
print("unscaled explained variance ratio:", raw.explained_variance_ratio_.round(3))
print(
    "loading of proline on PC1, unscaled:",
    raw.components_[0, list(Xw.columns).index("proline")].round(3),
)
print("scaled explained variance ratio:  ", scaled.explained_variance_ratio_.round(3))
print(
    "scaled PC1 loadings (largest three):",
    pd.Series(scaled.components_[0], Xw.columns).abs().nlargest(3).round(2).to_dict(),
)

Unscaled, PC1 is essentially the proline column (variance share 99.8%, loading near 1), which says nothing about the other twelve measurements. Standardise first (zero mean, unit variance) unless all features share a unit and their raw variance means something, as with pixels. Scaled, PC1 explains 36% and PC2 19%, and PC1 spreads its weight over several phenolic measurements (flavanoids, total phenols, OD280) instead of one column. Standardising turns PCA into an eigendecomposition of the correlation matrix.

**Nonlinear structure.** PCA finds straight directions. Concentric circles have no preferred straight direction at all.

In [ ]:
Xc_df, yc = make_circles_data(400, noise=0.03, seed=SEED)
Xc = Xc_df.to_numpy()
pc = PCA(2).fit(Xc)
print("explained variance ratio on two rings:", pc.explained_variance_ratio_.round(3))
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].scatter(*Xc.T, c=yc, s=10)
axes[0].set_title("two rings")
axes[1].scatter(pc.transform(Xc)[:, 0], np.zeros(len(Xc)), c=yc, s=10, alpha=0.4)
axes[1].set_title("PC1 only: the rings overlap completely")
plt.show()

Both directions have equal variance (50% each), so PCA has nothing to prefer, and projecting to one dimension piles the rings on top of each other. Notebook 05 covers nonlinear methods, and `KernelPCA` is the nonlinear extension of this notebook.

**Variance is not relevance.** PCA never sees a label. If the signal that separates two classes lives in a low-variance direction, PCA drops it.

In [ ]:
n = 600
labels = rng.integers(0, 2, n)
big_noise = rng.normal(0, 10, n)  # high variance, no information
signal = labels * 1.5 + rng.normal(0, 0.3, n)  # low variance, tells the classes apart
Xs = np.c_[big_noise, signal]
for k in (1, 2):
    m = make_pipeline(PCA(k, random_state=SEED), KNeighborsClassifier(5))
    print(f"PCA to {k} component(s): CV accuracy {cross_val_score(m, Xs, labels, cv=5).mean():.3f}")

With one component the model sees only the noise column and scores about 50%, which is chance. With both it is near perfect. When a supervised task is the goal, choose $k$ by validation accuracy, not by variance alone. Supervised alternatives such as linear discriminant analysis use the labels.

**Other limits.** Components are mixtures of every feature, so they are harder to explain than the originals. Outliers pull the components, because variance counts squared distances. Sign and order are the only things that are stable across runs, so fix a seed or check signs if you compare fits.

**Cost.** A full SVD is $O(nd^2)$ for $n>d$. For large data, `PCA(svd_solver="randomized")` finds the top components much faster, and `IncrementalPCA` handles data that do not fit in memory by processing batches.

Next, two methods built for drawing high-dimensional data in two dimensions: t-SNE and UMAP.